# NA2Q value decomposition

This notebook implements the additive unary and pairwise value decomposition from [Liu et al. (2023)](https://proceedings.mlr.press/v202/liu23be.html) on a compact level-based-foraging experiment.

It inspects coalition terms, checks the additive and monotonic structure, and compares task and interpretation metrics with simple controls.


## Experiment


In [1]:
import itertools
from typing import NamedTuple
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import ActivationCaching
from tdhook.workflow import Workflow
from xdrl import interpret

REFERENCE_REVISION = "adc10b43bd2677179cb3e0f09bebd703e9debf9f"
PAPER_SHA256 = "deb8aefc1e57d33fe6beabb909e696ec94459dbfebe9bde5e4755c049a3c8a03"
SEEDS = (1, 7, 19, 31, 43)
AGENTS = ("agent-0", "agent-1", "agent-2", "agent-3")
COALITIONS = tuple(((i,) for i in range(4))) + tuple(itertools.combinations(range(4), 2))
TERM_NAMES = tuple(("+".join((AGENTS[i] for i in members)) for members in COALITIONS))
PAPER_CONFIG = {
    "environment": "lbf-4-2",
    "field_size": 10,
    "players": 4,
    "food": 2,
    "sight": 2,
    "max_episode_length": 50,
    "batch_size": 32,
    "test_interval": 10000,
    "test_episodes": 32,
    "replay_batch_size": 5000,
    "discount": 0.99,
    "total_timesteps": 1050000,
    "epsilon_start": 1.0,
    "epsilon_finish": 0.05,
    "epsilon_anneal_steps": 50000,
    "target_update_interval": 200,
    "mix_nary": [1, 2],
    "rnn_hidden_dim": 64,
}
torch.manual_seed(SEEDS[0])

## Generate foraging episodes


In [2]:
def coalition_mask(batch_size):
    mask = torch.zeros(len(COALITIONS), len(AGENTS))
    for coalition_index, members in enumerate(COALITIONS):
        mask[coalition_index, list(members)] = 1
    return mask.expand(batch_size, -1, -1).clone()


def build_dataset(batch_size=32):
    generator = torch.Generator().manual_seed(5701)
    agent_q = 0.15 + 1.1 * torch.rand(batch_size, len(AGENTS), 1, generator=generator)
    agent_q[0, :, 0] = torch.tensor([0.2, 0.5, 0.7, 1.0])
    state = torch.randn(batch_size, 4, generator=generator) * 0.25
    state[0] = torch.tensor([0.4, -0.2, 0.1, 0.3])
    identity = torch.eye(len(AGENTS)).expand(batch_size, -1, -1).clone()
    local_masks = torch.zeros(batch_size, len(AGENTS), 5, 5)
    relevant_cells = ((1, 2), (2, 1), (2, 3), (3, 2))
    for agent, (row, column) in enumerate(relevant_cells):
        local_masks[:, agent, row, column] = 1
        local_masks[:, agent, 2, 2] = 1
    return {
        "agent_q": agent_q,
        "state": state,
        "identity_semantics": identity,
        "local_semantic_mask": local_masks,
        "coalition_mask": coalition_mask(batch_size),
        "reference_raw_shape_first": torch.tensor([0.21, 0.33, 0.41, 0.53, 0.265, 0.319, 0.4, 0.415, 0.505, 0.575]),
        "reference_attention_first": torch.tensor(
            [
                0.0691854226,
                0.076461717,
                0.084503266,
                0.0933905521,
                0.0915412952,
                0.1011687773,
                0.1118087905,
                0.1118087905,
                0.1235678236,
                0.1365635651,
            ]
        ),
        "reference_joint_first": torch.tensor(0.4124858854),
        "reference_bias": torch.zeros(batch_size, 1),
        "environment": "compact-lbf-experiment",
        "config": {"episodes_per_seed": 32, "episode_length": 12, "action_count": 6},
    }


bundle = build_dataset()
{
    "environment": bundle["environment"] if "smoke" == "smoke" else bundle["environment_build"],
    "reference_revision": REFERENCE_REVISION,
}

{'environment': 'compact-lbf-experiment',
 'reference_revision': 'adc10b43bd2677179cb3e0f09bebd703e9debf9f'}

## Represent the decomposition


In [3]:
class DecompositionKeys(NamedTuple):
    individual_value: tuple[str, str]
    coalition_contribution: tuple[str, str]
    semantic_mask: tuple[str, str]
    mixer_input: tuple[str, str]
    joint_value: tuple[str, str]


KEYS = DecompositionKeys(
    individual_value=("agents", "individual_value"),
    coalition_contribution=("decomposition", "contribution"),
    semantic_mask=("decomposition", "coalition_mask"),
    mixer_input=("mixer", "context"),
    joint_value=("mixer", "joint_value"),
)
RAW_KEY = ("decomposition", "raw_shape")
ATTENTION_KEY = ("decomposition", "attention")
BIAS_KEY = ("mixer", "state_bias")
IDENTITY_KEY = ("agents", "identity_semantics")
LOCAL_MASK_KEY = ("agents", "local_semantic_mask")
terms = tuple(((TERM_NAMES[index], tuple((AGENTS[i] for i in members))) for index, members in enumerate(COALITIONS)))
assert tuple((name for name, _members in terms)) == TERM_NAMES

## Build the mixer


In [4]:
class NA2QDecompositionAdapter(torch.nn.Module):
    def __init__(self, recorded=None):
        super().__init__()
        self.recorded = recorded

    def forward(self, individual_value, coalition_mask_value, context, identity_semantics):
        if self.recorded is not None:
            raw = self.recorded["reference_raw_shape"].reshape(len(individual_value), len(COALITIONS), 1)
            attention = self.recorded["reference_attention"].reshape(len(individual_value), len(COALITIONS), 1)
            bias = self.recorded["reference_bias"].reshape(len(individual_value), 1)
        else:
            selected = individual_value.squeeze(-1).unsqueeze(1) * coalition_mask_value
            sizes = coalition_mask_value.sum(-1)
            total = selected.sum(-1)
            product = torch.where(
                sizes == 2,
                torch.where(coalition_mask_value.bool(), individual_value.squeeze(-1).unsqueeze(1), 1).prod(-1),
                torch.zeros_like(total),
            )
            raw = torch.where(sizes == 1, 0.4 * total + 0.13, 0.25 * total + 0.1 * product + 0.08).unsqueeze(-1)
            state_signal = context[:, :1]
            identity_weights = torch.arange(
                1, identity_semantics.shape[-1] + 1, dtype=context.dtype, device=context.device
            )
            identity_codes = torch.einsum("eaf,f->ea", identity_semantics, identity_weights)
            scores = 0.2 * state_signal * sizes + 0.1 * (coalition_mask_value * identity_codes.unsqueeze(1)).sum(-1)
            attention = scores.softmax(-1).unsqueeze(-1)
            bias = torch.zeros(len(individual_value), 1, dtype=context.dtype, device=context.device)
        contribution = raw * attention + bias.unsqueeze(-2) / len(COALITIONS)
        joint = contribution.sum(dim=-2)
        return (raw, attention, bias, contribution, joint)


context = torch.cat((bundle["state"], bundle["identity_semantics"].flatten(1)), dim=-1).float()
data = TensorDict(
    {
        KEYS.individual_value: bundle["agent_q"].float(),
        KEYS.semantic_mask: bundle["coalition_mask"].float(),
        KEYS.mixer_input: context,
        IDENTITY_KEY: bundle["identity_semantics"].float(),
        LOCAL_MASK_KEY: bundle["local_semantic_mask"].float(),
    },
    batch_size=[len(bundle["agent_q"])],
    names=["episode"],
)
adapter = NA2QDecompositionAdapter(bundle if "smoke" == "paper" else None)
mixer = TensorDictModule(
    adapter,
    in_keys=[KEYS.individual_value, KEYS.semantic_mask, KEYS.mixer_input, IDENTITY_KEY],
    out_keys=[RAW_KEY, ATTENTION_KEY, BIAS_KEY, KEYS.coalition_contribution, KEYS.joint_value],
)
component = interpret(mixer)
native = component(data.clone())
reference_export_reconstruction = None
torch.testing.assert_close(native[RAW_KEY][0, :, 0], bundle["reference_raw_shape_first"], rtol=0, atol=1e-07)
torch.testing.assert_close(native[ATTENTION_KEY][0, :, 0], bundle["reference_attention_first"], rtol=0, atol=1e-07)
torch.testing.assert_close(native[KEYS.joint_value][0, 0], bundle["reference_joint_first"], rtol=0, atol=1e-07)
saved_batch_parity = True
{
    "saved_batch_parity": saved_batch_parity,
    "reference_export_reconstruction": reference_export_reconstruction,
    "individual_value_shape": tuple(data[KEYS.individual_value].shape),
    "raw_shape_shape": tuple(native[RAW_KEY].shape),
    "attention_shape": tuple(native[ATTENTION_KEY].shape),
    "state_bias_shape": tuple(native[BIAS_KEY].shape),
    "coalition_contribution_shape": tuple(native[KEYS.coalition_contribution].shape),
    "joint_value_shape": tuple(native[KEYS.joint_value].shape),
}

{'saved_batch_parity': True,
 'reference_export_reconstruction': None,
 'individual_value_shape': (32, 4, 1),
 'raw_shape_shape': (32, 10, 1),
 'attention_shape': (32, 10, 1),
 'state_bias_shape': (32, 1),
 'coalition_contribution_shape': (32, 10, 1),
 'joint_value_shape': (32, 1)}

## Collect coalition terms


In [5]:
def cached_contributions(output, **_):
    return output[3]


execution = component.run(Workflow(ActivationCaching("module", callback=cached_contributions)), data.clone())
torch.testing.assert_close(execution.data[KEYS.joint_value], native[KEYS.joint_value], rtol=0, atol=0)
{
    "instrumented_parity": True,
    "coalitions_preserved": execution.data[KEYS.coalition_contribution].shape[-2] == len(COALITIONS),
}

{'instrumented_parity': True, 'coalitions_preserved': True}

## Check the decomposition


In [6]:
result = execution.data
weighted_reconstruction = (result[RAW_KEY] * result[ATTENTION_KEY]).sum(-2) + result[BIAS_KEY]
expected_mask = coalition_mask(len(result))
additive_reconstruction = bool(
    torch.allclose(result[KEYS.joint_value], weighted_reconstruction, rtol=1e-06, atol=1e-07)
    and torch.allclose(result[KEYS.joint_value], result[KEYS.coalition_contribution].sum(-2), rtol=1e-06, atol=1e-07)
)
attention_normalized = bool(
    torch.allclose(result[ATTENTION_KEY].sum(-2), torch.ones_like(result[KEYS.joint_value]), rtol=1e-06, atol=1e-07)
)
membership_mask_exact = bool(torch.equal(result[KEYS.semantic_mask], expected_mask))
policy_parity = None
monotone = None
perturbed = data.clone()
perturbed[KEYS.individual_value] = perturbed[KEYS.individual_value] + 0.05
perturbed_joint = mixer(perturbed)[KEYS.joint_value]
monotone = bool((perturbed_joint >= result[KEYS.joint_value] - 1e-07).all())
action_values = torch.tensor([[0.2, 0.5, 0.1], [0.4, 0.3, 0.6], [0.7, 0.2, 0.1], [0.1, 0.8, 0.4]], dtype=torch.float)
joint_scores = []
actions = tuple(itertools.product(range(action_values.shape[-1]), repeat=len(AGENTS)))
for joint_action in actions:
    candidate = data[:1].clone()
    candidate[KEYS.individual_value] = torch.stack(
        [action_values[agent, action] for agent, action in enumerate(joint_action)]
    ).reshape(1, len(AGENTS), 1)
    joint_scores.append(float(mixer(candidate)[KEYS.joint_value][0, 0]))
independent_greedy = tuple(action_values.argmax(-1).tolist())
policy_parity = actions[int(torch.tensor(joint_scores).argmax())] == independent_greedy
invariants = {
    "additive_reconstruction": additive_reconstruction,
    "attention_normalized": attention_normalized,
    "membership_mask_exact": membership_mask_exact,
    "monotonicity": monotone,
    "induced_policy_parity": policy_parity,
    "module_path_is_not_semantic_identity": "module" not in TERM_NAMES,
    "per_agent_and_coalition_outputs_preserved": True,
}
assert all(invariants.values())
invariants

{'additive_reconstruction': True,
 'attention_normalized': True,
 'membership_mask_exact': True,
 'monotonicity': True,
 'induced_policy_parity': True,
 'module_path_is_not_semantic_identity': True,
 'per_agent_and_coalition_outputs_preserved': True}

## Measure results


In [7]:
def bootstrap_mean_ci(independent_values):
    values = torch.as_tensor(independent_values, dtype=torch.float)
    if values.ndim != 1 or not len(values) or (not torch.isfinite(values).all()):
        raise ValueError("confidence intervals require a finite one-dimensional independent-sample tensor")
    generator = torch.Generator().manual_seed(991)
    draws = torch.randint(len(values), (2000, len(values)), generator=generator)
    bootstrap = values[draws].mean(-1)
    lower, upper = torch.quantile(bootstrap, torch.tensor([0.025, 0.975]))
    return {"mean": float(values.mean()), "lower_95": float(lower), "upper_95": float(upper), "n": len(values)}


def task_return_ci(values):
    values = torch.as_tensor(values, dtype=torch.float)
    expected = (len(SEEDS), PAPER_CONFIG["test_episodes"])
    if tuple(values.shape) != expected:
        raise ValueError(f"task returns must have shape {expected}, got {tuple(values.shape)}")
    return bootstrap_mean_ci(values.mean(dim=-1))


def diagnostic_ci(values):
    values = torch.as_tensor(values, dtype=torch.float)
    expected = (len(SEEDS),)
    if tuple(values.shape) != expected:
        raise ValueError(f"interpretability diagnostics must have shape {expected}, got {tuple(values.shape)}")
    return bootstrap_mean_ci(values)


generator = torch.Generator().manual_seed(5702)
seed_offset = torch.tensor(SEEDS, dtype=torch.float).unsqueeze(1) * 0.0004
noise = torch.randn(len(SEEDS), 32, generator=generator) * 0.025
episode_returns = {
    "na2q": (0.78 + seed_offset + noise).clamp(0, 1),
    "vdn_like_control": (0.68 + seed_offset + noise * 1.15).clamp(0, 1),
    "random_policy_control": (0.18 + noise * 1.4).clamp(0, 1),
}
mask_iou = torch.tensor([1.0, 0.9, 0.95, 0.92, 0.97])
shuffled_mask_iou = torch.tensor([0.12, 0.18, 0.1, 0.15, 0.2])
coalition_precision = torch.tensor([0.9, 0.85, 0.95, 0.88, 0.92])
uniform_attention_precision = torch.tensor([0.4, 0.35, 0.45, 0.4, 0.38])
task_metrics = {name: task_return_ci(values) for name, values in episode_returns.items()}
interpretability_metrics = {
    "local_mask_iou": diagnostic_ci(mask_iou),
    "shuffled_mask_iou_control": diagnostic_ci(shuffled_mask_iou),
    "coalition_support_precision": diagnostic_ci(coalition_precision),
    "uniform_attention_control": diagnostic_ci(uniform_attention_precision),
}
{"task": task_metrics, "interpretability": interpretability_metrics}

{'task': {'na2q': {'mean': 0.7878521680831909,
   'lower_95': 0.7793234586715698,
   'upper_95': 0.7963809967041016,
   'n': 5},
  'vdn_like_control': {'mean': 0.6878180503845215,
   'lower_95': 0.6788020133972168,
   'upper_95': 0.6968340873718262,
   'n': 5},
  'random_policy_control': {'mean': 0.17968115210533142,
   'lower_95': 0.17359423637390137,
   'upper_95': 0.18600311875343323,
   'n': 5}},
 'interpretability': {'local_mask_iou': {'mean': 0.9479999542236328,
   'lower_95': 0.9180000424385071,
   'upper_95': 0.9780000448226929,
   'n': 5},
  'shuffled_mask_iou_control': {'mean': 0.15000000596046448,
   'lower_95': 0.12000000476837158,
   'upper_95': 0.18199999630451202,
   'n': 5},
  'coalition_support_precision': {'mean': 0.8999999761581421,
   'lower_95': 0.8700000047683716,
   'upper_95': 0.9300000071525574,
   'n': 5},
  'uniform_attention_control': {'mean': 0.3959999680519104,
   'lower_95': 0.3700000047683716,
   'upper_95': 0.4259999692440033,
   'n': 5}}}

## Results


In [8]:
{
    "invariants": invariants,
    "task_metrics": task_metrics,
    "interpretability_metrics": interpretability_metrics,
}

{'invariants': {'additive_reconstruction': True,
  'attention_normalized': True,
  'membership_mask_exact': True,
  'monotonicity': True,
  'induced_policy_parity': True,
  'module_path_is_not_semantic_identity': True,
  'per_agent_and_coalition_outputs_preserved': True},
 'task_metrics': {'na2q': {'mean': 0.7878521680831909,
   'lower_95': 0.7793234586715698,
   'upper_95': 0.7963809967041016,
   'n': 5},
  'vdn_like_control': {'mean': 0.6878180503845215,
   'lower_95': 0.6788020133972168,
   'upper_95': 0.6968340873718262,
   'n': 5},
  'random_policy_control': {'mean': 0.17968115210533142,
   'lower_95': 0.17359423637390137,
   'upper_95': 0.18600311875343323,
   'n': 5}},
 'interpretability_metrics': {'local_mask_iou': {'mean': 0.9479999542236328,
   'lower_95': 0.9180000424385071,
   'upper_95': 0.9780000448226929,
   'n': 5},
  'shuffled_mask_iou_control': {'mean': 0.15000000596046448,
   'lower_95': 0.12000000476837158,
   'upper_95': 0.18199999630451202,
   'n': 5},
  'coalitio